In [ ]:
# ==============================================================================
# CELL 1: TRAINING ALL SPECIALIZED MODELS PER LEAGUE
# ==============================================================================
import pandas as pd
import glob
import numpy as np
import joblib
import os
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

# --- All function definitions (no changes here) ---
def expected_outcome(rating1, rating2): return 1 / (1 + 10 ** ((rating2 - rating1) / 400))
def update_elo(home_elo, away_elo, fthg, ftag, k=20):
    e_home = expected_outcome(home_elo, away_elo)
    s_home = 1 if fthg > ftag else 0 if ftag > fthg else 0.5
    updated_home_elo = home_elo + k * (s_home - e_home)
    updated_away_elo = away_elo - (updated_home_elo - home_elo)
    return updated_home_elo, updated_away_elo
def calculate_elo_for_league(df, initial_elo=1500, k=20):
    elo_ratings = {}
    home_elos, away_elos = [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        home_elo, away_elo = elo_ratings.get(home_team, initial_elo), elo_ratings.get(away_team, initial_elo)
        home_elos.append(home_elo); away_elos.append(away_elo)
        new_home_elo, new_away_elo = update_elo(home_elo, away_elo, row['FTHG'], row['FTAG'], k)
        elo_ratings[home_team], elo_ratings[away_team] = new_home_elo, new_away_elo
    df['HomeElo'], df['AwayElo'] = home_elos, away_elos
    df['FinalHomeElo'] = df['HomeTeam'].map(elo_ratings)
    df['FinalAwayElo'] = df['AwayTeam'].map(elo_ratings)
    return df
def get_form_points(df, n_matches=5):
    team_matches = {}
    for _, row in df.iterrows():
        res = row['FTR']
        if res == 'H': home_pts, away_pts = 3, 0
        elif res == 'A': home_pts, away_pts = 0, 3
        else: home_pts, away_pts = 1, 1
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_matches: team_matches[home_team] = []
        if away_team not in team_matches: team_matches[away_team] = []
        team_matches[home_team].append(home_pts)
        team_matches[away_team].append(away_pts)
    home_form, away_form = [], []
    for _, row in df.iterrows():
        home_form.append(sum(team_matches[row['HomeTeam']][-n_matches-1:-1]))
        away_form.append(sum(team_matches[row['AwayTeam']][-n_matches-1:-1]))
    df['HomeForm'], df['AwayForm'] = home_form, away_form
    return df
def get_goal_averages(df):
    team_stats = {}
    home_goals_avg, away_goals_avg, home_conceded_avg, away_conceded_avg = [], [], [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_stats: team_stats[home_team] = {'scored': [], 'conceded': []}
        if away_team not in team_stats: team_stats[away_team] = {'scored': [], 'conceded': []}
        home_goals_avg.append(np.mean(team_stats[home_team]['scored']) if team_stats[home_team]['scored'] else 0)
        home_conceded_avg.append(np.mean(team_stats[home_team]['conceded']) if team_stats[home_team]['conceded'] else 0)
        away_goals_avg.append(np.mean(team_stats[away_team]['scored']) if team_stats[away_team]['scored'] else 0)
        away_conceded_avg.append(np.mean(team_stats[away_team]['conceded']) if team_stats[away_team]['conceded'] else 0)
        team_stats[home_team]['scored'].append(row['FTHG']); team_stats[home_team]['conceded'].append(row['FTAG'])
        team_stats[away_team]['scored'].append(row['FTAG']); team_stats[away_team]['conceded'].append(row['FTHG'])
    df['AvgHomeGoals'], df['AvgHomeConceded'] = home_goals_avg, home_conceded_avg
    df['AvgAwayGoals'], df['AvgAwayConceded'] = away_goals_avg, away_conceded_avg
    return df
def create_features(df):
    df['result_label'] = df['FTR'].map({'H': 2, 'D': 1, 'A': 0})
    df['over_2_5'] = ((df['FTHG'] + df['FTAG']) > 2.5).astype(int)
    df['btts'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    df['Elo_diff'] = df['HomeElo'] - df['AwayElo']
    df = get_form_points(df)
    df = get_goal_averages(df)
    df = df.dropna(subset=['result_label'])
    df['result_label'] = df['result_label'].astype(int)
    return df
def load_and_process_all_leagues(base_path='data/'):
    all_files = glob.glob(f'{base_path}/**/*.csv', recursive=True)
    all_league_dfs = []
    print(f"Found {len(all_files)} files to process.")
    for f in all_files:
        try:
            league_name = os.path.basename(os.path.dirname(f))
            # Skip files where league_name is invalid (e.g., 'data' or other non-league folders)
            if league_name in ['data', 'global', 'DANSK']:
                continue
            delimiter = ';' if any(x in f for x in ['CHN', 'JPN', 'CH/']) else ','
            temp_df = pd.read_csv(f, delimiter=delimiter, on_bad_lines='skip', encoding='latin1')
            temp_df['league'] = league_name
            temp_df = temp_df.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam', 'HG': 'FTHG', 'AG': 'FTAG', 'Res': 'FTR'})
            required_cols = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Date']
            if not all(col in temp_df.columns for col in required_cols): continue
            temp_df['Date'] = pd.to_datetime(temp_df['Date'], dayfirst=True, errors='coerce')
            temp_df = temp_df.dropna(subset=['Date', 'FTHG', 'FTAG']).sort_values('Date').reset_index(drop=True)
            df_with_elo = calculate_elo_for_league(temp_df)
            df_featured = create_features(df_with_elo)
            all_league_dfs.append(df_featured)
        except Exception as e:
            print(f"     ... ERROR processing file {f}: {e}")
    return pd.concat(all_league_dfs, ignore_index=True).sort_values('Date').reset_index(drop=True)

# --- Main Training Execution ---
all_data = load_and_process_all_leagues(base_path='data/')
all_data = all_data[all_data['Date'].dt.year >= 2020].copy()
all_data = all_data.rename(columns={'B365>2.5': 'B365_O2_5', 'B365<2.5': 'B365_U2_5'})
print(f"\nTotal historical matches (2020 onwards) to be used: {len(all_data)}")

# --- Define Feature Lists and Param Grid ---
feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
leagues = all_data['league'].unique()
param_grid = {
    'n_estimators': list(range(801, 2851, 2)) + list(range(3001,3300,2)),
    'max_depth': [3],
    'learning_rate': [0.1,0.01,0.005],
}

# --- Train 1X2 Models ---
for league in leagues:
    print(f"\n--- Training 1X2 model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['result_label'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['result_label']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='multi:softprob'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_1x2_{league}.joblib')

# --- Train O/U 2.5 Models ---
for league in leagues:
    print(f"\n--- Training O/U 2.5 model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['over_2_5'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['over_2_5']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='binary:logistic'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_ou25_{league}.joblib')

# --- Train BTTS Models ---
for league in leagues:
    print(f"\n--- Training BTTS model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['btts'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['btts']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='binary:logistic'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_btts_{league}.joblib')

all_data.to_csv('full_processed_data.csv', index=False)
print("\n--- All specialized models for all markets have been trained and saved. ---")

Found 69 files to process.

Total historical matches (2020 onwards) to be used: 32229

--- Training 1X2 model for Ch ---

--- Training 1X2 model for Japan ---

--- Training 1X2 model for Russian ---

--- Training 1X2 model for Swedish ---

--- Training 1X2 model for Norsk ---

--- Training 1X2 model for Fin ---

--- Training 1X2 model for Chn ---

--- Training 1X2 model for French ---

--- Training 1X2 model for British_Champ ---

--- Training 1X2 model for Turkish ---

--- Training 1X2 model for British_Pl ---

--- Training 1X2 model for Dutch ---

--- Training 1X2 model for Spanish_2 ---

--- Training 1X2 model for Spanish ---

--- Training 1X2 model for German ---

--- Training 1X2 model for German_2 ---

--- Training 1X2 model for Porto ---

--- Training 1X2 model for Italian ---

--- Training 1X2 model for British_Conference ---

--- Training O/U 2.5 model for Ch ---

--- Training O/U 2.5 model for Japan ---

--- Training O/U 2.5 model for Russian ---

--- Training O/U 2.5 model f

In [ ]:
# ==============================================================================
# CELL TO TRAIN EXPECTED GOALS (xG) MODELS PER LEAGUE
# ==============================================================================
import pandas as pd
import glob
import numpy as np
import joblib
import os
from xgboost import XGBRegressor # Note: We use XGBRegressor for predicting numbers
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# --- All function definitions (no changes here) ---
def expected_outcome(rating1, rating2): return 1 / (1 + 10 ** ((rating2 - rating1) / 400))
def update_elo(home_elo, away_elo, fthg, ftag, k=20):
    e_home = expected_outcome(home_elo, away_elo)
    s_home = 1 if fthg > ftag else 0 if ftag > fthg else 0.5
    updated_home_elo = home_elo + k * (s_home - e_home)
    updated_away_elo = away_elo - (updated_home_elo - home_elo)
    return updated_home_elo, updated_away_elo
def calculate_elo_for_league(df, initial_elo=1500, k=20):
    elo_ratings = {}
    home_elos, away_elos = [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        home_elo, away_elo = elo_ratings.get(home_team, initial_elo), elo_ratings.get(away_team, initial_elo)
        home_elos.append(home_elo); away_elos.append(away_elo)
        new_home_elo, new_away_elo = update_elo(home_elo, away_elo, row['FTHG'], row['FTAG'], k)
        elo_ratings[home_team], elo_ratings[away_team] = new_home_elo, new_away_elo
    df['HomeElo'], df['AwayElo'] = home_elos, away_elos
    df['FinalHomeElo'] = df['HomeTeam'].map(elo_ratings)
    df['FinalAwayElo'] = df['AwayTeam'].map(elo_ratings)
    return df
def get_form_points(df, n_matches=5):
    team_matches = {}
    for _, row in df.iterrows():
        res = row['FTR']
        if res == 'H': home_pts, away_pts = 3, 0
        elif res == 'A': home_pts, away_pts = 0, 3
        else: home_pts, away_pts = 1, 1
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_matches: team_matches[home_team] = []
        if away_team not in team_matches: team_matches[away_team] = []
        team_matches[home_team].append(home_pts)
        team_matches[away_team].append(away_pts)
    home_form, away_form = [], []
    for _, row in df.iterrows():
        home_form.append(sum(team_matches[row['HomeTeam']][-n_matches-1:-1]))
        away_form.append(sum(team_matches[row['AwayTeam']][-n_matches-1:-1]))
    df['HomeForm'], df['AwayForm'] = home_form, away_form
    return df
def get_goal_averages(df):
    team_stats = {}
    home_goals_avg, away_goals_avg, home_conceded_avg, away_conceded_avg = [], [], [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_stats: team_stats[home_team] = {'scored': [], 'conceded': []}
        if away_team not in team_stats: team_stats[away_team] = {'scored': [], 'conceded': []}
        home_goals_avg.append(np.mean(team_stats[home_team]['scored']) if team_stats[home_team]['scored'] else 0)
        home_conceded_avg.append(np.mean(team_stats[home_team]['conceded']) if team_stats[home_team]['conceded'] else 0)
        away_goals_avg.append(np.mean(team_stats[away_team]['scored']) if team_stats[away_team]['scored'] else 0)
        away_conceded_avg.append(np.mean(team_stats[away_team]['conceded']) if team_stats[away_team]['conceded'] else 0)
        team_stats[home_team]['scored'].append(row['FTHG']); team_stats[home_team]['conceded'].append(row['FTAG'])
        team_stats[away_team]['scored'].append(row['FTAG']); team_stats[away_team]['conceded'].append(row['FTHG'])
    df['AvgHomeGoals'], df['AvgHomeConceded'] = home_goals_avg, home_conceded_avg
    df['AvgAwayGoals'], df['AvgAwayConceded'] = away_goals_avg, away_conceded_avg
    return df
def create_features(df):
    df['Elo_diff'] = df['HomeElo'] - df['AwayElo']
    df = get_form_points(df)
    df = get_goal_averages(df)
    return df
def load_and_process_all_leagues(base_path='data/'):
    all_files = glob.glob(f'{base_path}/**/*.csv', recursive=True)
    all_league_dfs = []
    print(f"Found {len(all_files)} files to process.")
    for f in all_files:
        try:
            league_name = os.path.basename(os.path.dirname(f))
            # Skip files where league_name is invalid (e.g., 'data' or other non-league folders)
            if league_name in ['data', 'global', 'DANSK']:
                continue
            delimiter = ';' if any(x in f for x in ['CHN', 'JPN', 'CH/']) else ','
            temp_df = pd.read_csv(f, delimiter=delimiter, on_bad_lines='skip', encoding='latin1')
            temp_df['league'] = league_name
            temp_df = temp_df.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam', 'HG': 'FTHG', 'AG': 'FTAG', 'Res': 'FTR'})
            required_cols = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Date']
            if not all(col in temp_df.columns for col in required_cols): continue
            temp_df['Date'] = pd.to_datetime(temp_df['Date'], dayfirst=True, errors='coerce')
            temp_df = temp_df.dropna(subset=['Date', 'FTHG', 'FTAG']).sort_values('Date').reset_index(drop=True)
            df_with_elo = calculate_elo_for_league(temp_df)
            df_featured = create_features(df_with_elo)
            all_league_dfs.append(df_featured)
        except Exception as e:
            print(f"     ... ERROR processing file {f}: {e}")
    return pd.concat(all_league_dfs, ignore_index=True).sort_values('Date').reset_index(drop=True)

# --- Main Training Execution ---
all_data = load_and_process_all_leagues(base_path='data/')
all_data = all_data[all_data['Date'].dt.year >= 2020].copy()
print(f"\nTotal historical matches (2020 onwards) to be used: {len(all_data)}")

# --- Define Feature Lists and Param Grid for Regression ---
feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
leagues = all_data['league'].unique()
param_grid = {
    'n_estimators': list(range(750, 3301, 2)),
    'max_depth': [3],
    'learning_rate': [0.01, 0.05],
}

# --- Train Home Expected Goals (xGH) Models ---
for league in leagues:
    print(f"\n--- Training xG Home model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['FTHG'])
    if len(league_data) < 100: continue
    
    X_train, y_train = league_data[feature_cols], league_data['FTHG']
    
    grid_search = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror'), # Use XGBRegressor for this task
        param_grid=param_grid, 
        scoring='neg_mean_squared_error', # A common scoring method for regression
        cv=3, 
        verbose=1,
        n_jobs=-1
    ).fit(X_train, y_train)
    
    print(f"  Best params for {league.title()} xGH: {grid_search.best_params_}")
    joblib.dump(grid_search.best_estimator_, f'model_xGH_{league}.joblib')

# --- Train Away Expected Goals (xGA) Models ---
for league in leagues:
    print(f"\n--- Training xG Away model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['FTAG'])
    if len(league_data) < 100: continue
        
    X_train, y_train = league_data[feature_cols], league_data['FTAG']
    
    grid_search = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror'),
        param_grid=param_grid, 
        scoring='neg_mean_squared_error',
        cv=3, 
        verbose=1,
        n_jobs=-1
    ).fit(X_train, y_train)
    
    print(f"  Best params for {league.title()} xGA: {grid_search.best_params_}")
    joblib.dump(grid_search.best_estimator_, f'model_xGA_{league}.joblib')

print("\n--- All specialized xG models have been trained and saved. ---")

Found 84 files to process.

Total historical matches (2020 onwards) to be used: 36196

--- Training xG Home model for Ch ---
Fitting 3 folds for each of 2552 candidates, totalling 7656 fits
  Best params for Ch xGH: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 750}

--- Training xG Home model for Japan ---
Fitting 3 folds for each of 2552 candidates, totalling 7656 fits
  Best params for Japan xGH: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 750}

--- Training xG Home model for Russian ---
Fitting 3 folds for each of 2552 candidates, totalling 7656 fits
  Best params for Russian xGH: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 750}

--- Training xG Home model for Swedish ---
Fitting 3 folds for each of 2552 candidates, totalling 7656 fits
  Best params for Swedish xGH: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 750}

--- Training xG Home model for Norsk ---
Fitting 3 folds for each of 2552 candidates, totalling 7656 fits
  Best params for

In [4]:
"""STANDALONE OLDEST AND NEWEST MATCHES"""

import pandas as pd

try:
    # Load the final dataset that the models were trained on
    df = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    
    print("--- Training Data Date Range Per League ---")
    
    # Get a list of all unique leagues in your dataset
    leagues = sorted(df['league'].unique())
    
    # Loop through each league and print its date range
    for league in leagues:
        # Create a temporary dataframe for only the current league
        league_df = df[df['league'] == league]
        
        # Find the minimum (oldest) and maximum (newest) dates
        oldest_date = league_df['Date'].min()
        newest_date = league_df['Date'].max()
        
        print(f"\nLeague: {league.title()}")
        print(f"  Oldest match: {oldest_date.strftime('%Y-%m-%d')}")
        print(f"  Newest match: {newest_date.strftime('%Y-%m-%d')}")

except FileNotFoundError:
    print("Could not find 'full_processed_data.csv'. Please run the main training cell first.")

--- Training Data Date Range Per League ---

League: Ch
  Oldest match: 2012-07-13
  Newest match: 2025-05-24

League: Belgian
  Oldest match: 2020-08-08
  Newest match: 2025-10-26

League: British_Champ
  Oldest match: 2020-09-11
  Newest match: 2025-05-03

League: British_Conference
  Oldest match: 2020-10-03
  Newest match: 2025-05-05

League: British_Pl
  Oldest match: 2020-09-12
  Newest match: 2025-11-09

League: Bundesliga_2
  Oldest match: 2021-07-23
  Newest match: 2025-05-18

League: Chn
  Oldest match: 2014-03-07
  Newest match: 2025-06-30

League: Data
  Oldest match: 2025-10-30
  Newest match: 2025-11-09

League: Dutch
  Oldest match: 2020-09-12
  Newest match: 2025-10-26

League: Fin
  Oldest match: 2012-04-15
  Newest match: 2025-07-07

League: French
  Oldest match: 2020-08-21
  Newest match: 2025-10-29

League: German
  Oldest match: 2020-09-18
  Newest match: 2025-11-09

League: German_2
  Oldest match: 2020-09-18
  Newest match: 2025-10-05

League: Greek
  Oldest mat

In [ ]:
# ==============================================================================
# CELL 1: TRAINING MODELS AND VALIDATING DATE RANGES
# ==============================================================================
import pandas as pd
import glob
import numpy as np
import joblib
import os
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

# --- All function definitions (no changes here) ---
def expected_outcome(rating1, rating2): return 1 / (1 + 10 ** ((rating2 - rating1) / 400))
def update_elo(home_elo, away_elo, fthg, ftag, k=20):
    e_home = expected_outcome(home_elo, away_elo)
    s_home = 1 if fthg > ftag else 0 if ftag > fthg else 0.5
    updated_home_elo = home_elo + k * (s_home - e_home)
    updated_away_elo = away_elo - (updated_home_elo - home_elo)
    return updated_home_elo, updated_away_elo
def calculate_elo_for_league(df, initial_elo=1500, k=20):
    elo_ratings = {}
    home_elos, away_elos = [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        home_elo, away_elo = elo_ratings.get(home_team, initial_elo), elo_ratings.get(away_team, initial_elo)
        home_elos.append(home_elo); away_elos.append(away_elo)
        new_home_elo, new_away_elo = update_elo(home_elo, away_elo, row['FTHG'], row['FTAG'], k)
        elo_ratings[home_team], elo_ratings[away_team] = new_home_elo, new_away_elo
    df['HomeElo'], df['AwayElo'] = home_elos, away_elos
    df['FinalHomeElo'] = df['HomeTeam'].map(elo_ratings)
    df['FinalAwayElo'] = df['AwayTeam'].map(elo_ratings)
    return df
def get_form_points(df, n_matches=5):
    team_matches = {}
    for _, row in df.iterrows():
        res = row['FTR']
        if res == 'H': home_pts, away_pts = 3, 0
        elif res == 'A': home_pts, away_pts = 0, 3
        else: home_pts, away_pts = 1, 1
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_matches: team_matches[home_team] = []
        if away_team not in team_matches: team_matches[away_team] = []
        team_matches[home_team].append(home_pts)
        team_matches[away_team].append(away_pts)
    home_form, away_form = [], []
    for _, row in df.iterrows():
        home_form.append(sum(team_matches[row['HomeTeam']][-n_matches-1:-1]))
        away_form.append(sum(team_matches[row['AwayTeam']][-n_matches-1:-1]))
    df['HomeForm'], df['AwayForm'] = home_form, away_form
    return df
def get_goal_averages(df):
    team_stats = {}
    home_goals_avg, away_goals_avg, home_conceded_avg, away_conceded_avg = [], [], [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_stats: team_stats[home_team] = {'scored': [], 'conceded': []}
        if away_team not in team_stats: team_stats[away_team] = {'scored': [], 'conceded': []}
        home_goals_avg.append(np.mean(team_stats[home_team]['scored']) if team_stats[home_team]['scored'] else 0)
        home_conceded_avg.append(np.mean(team_stats[home_team]['conceded']) if team_stats[home_team]['conceded'] else 0)
        away_goals_avg.append(np.mean(team_stats[away_team]['scored']) if team_stats[away_team]['scored'] else 0)
        away_conceded_avg.append(np.mean(team_stats[away_team]['conceded']) if team_stats[away_team]['conceded'] else 0)
        team_stats[home_team]['scored'].append(row['FTHG']); team_stats[home_team]['conceded'].append(row['FTAG'])
        team_stats[away_team]['scored'].append(row['FTAG']); team_stats[away_team]['conceded'].append(row['FTHG'])
    df['AvgHomeGoals'], df['AvgHomeConceded'] = home_goals_avg, home_conceded_avg
    df['AvgAwayGoals'], df['AvgAwayConceded'] = away_goals_avg, away_conceded_avg
    return df
def create_features(df):
    df['result_label'] = df['FTR'].map({'H': 2, 'D': 1, 'A': 0})
    df['over_2_5'] = ((df['FTHG'] + df['FTAG']) > 2.5).astype(int)
    df['btts'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    df['Elo_diff'] = df['HomeElo'] - df['AwayElo']
    df = get_form_points(df)
    df = get_goal_averages(df)
    df = df.dropna(subset=['result_label'])
    df['result_label'] = df['result_label'].astype(int)
    return df
def load_and_process_all_leagues(base_path='data/'):
    all_files = glob.glob(f'{base_path}/**/*.csv', recursive=True)
    all_league_dfs = []
    print(f"Found {len(all_files)} files to process.")
    for f in all_files:
        try:
            league_name = os.path.basename(os.path.dirname(f))
            # Skip files where league_name is invalid (e.g., 'data' or other non-league folders)
            if league_name in ['data', 'global', 'DANSK']:
                continue
            delimiter = ';' if any(x in f for x in ['CHN', 'JPN', 'CH/']) else ','
            temp_df = pd.read_csv(f, delimiter=delimiter, on_bad_lines='skip', encoding='latin1')
            temp_df['league'] = league_name
            temp_df = temp_df.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam', 'HG': 'FTHG', 'AG': 'FTAG', 'Res': 'FTR'})
            required_cols = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Date']
            if not all(col in temp_df.columns for col in required_cols): continue
            temp_df['Date'] = pd.to_datetime(temp_df['Date'], dayfirst=True, errors='coerce')
            temp_df = temp_df.dropna(subset=['Date', 'FTHG', 'FTAG']).sort_values('Date').reset_index(drop=True)
            df_with_elo = calculate_elo_for_league(temp_df)
            df_featured = create_features(df_with_elo)
            all_league_dfs.append(df_featured)
        except Exception as e:
            print(f"     ... ERROR processing file {f}: {e}")
    return pd.concat(all_league_dfs, ignore_index=True).sort_values('Date').reset_index(drop=True)

# --- Main Training Execution ---
all_data = load_and_process_all_leagues(base_path='data/')
all_data = all_data[all_data['Date'].dt.year >= 2020].copy()
all_data = all_data.rename(columns={'B365>2.5': 'B365_O2_5', 'B365<2.5': 'B365_U2_5'})
print(f"\nTotal historical matches (2020 onwards) to be used: {len(all_data)}")

# --- Define Feature Lists and Param Grid ---
feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
leagues = all_data['league'].unique()
param_grid = {
    'n_estimators': list(range(1000, 1051, 1)), # A smaller range for quicker execution
    'max_depth': [3],
    'learning_rate': [0.01],
}

# --- Train 1X2 Models ---
for league in leagues:
    print(f"\n--- Training 1X2 model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['result_label'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['result_label']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='multi:softprob'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_1x2_{league}.joblib')

# --- Train O/U 2.5 Models ---
for league in leagues:
    print(f"\n--- Training O/U 2.5 model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['over_2_5'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['over_2_5']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='binary:logistic'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_ou25_{league}.joblib')

# --- Train BTTS Models ---
for league in leagues:
    print(f"\n--- Training BTTS model for {league.title()} ---")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['btts'])
    if len(league_data) < 100: continue
    X_train, y_train = league_data[feature_cols], league_data['btts']
    grid_search = GridSearchCV(estimator=XGBClassifier(objective='binary:logistic'), param_grid=param_grid, cv=3, n_jobs=-1).fit(X_train, y_train)
    calibrated_model = CalibratedClassifierCV(estimator=grid_search.best_estimator_, cv='prefit').fit(X_train, y_train)
    joblib.dump(calibrated_model, f'model_btts_{league}.joblib')

all_data.to_csv('full_processed_data.csv', index=False)
print("\n--- All specialized models for all markets have been trained and saved. ---")

# ==============================================================================
# NEW: EMBEDDED DATE VALIDATION SCRIPT
# ==============================================================================
try:
    df_check = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    
    print("\n\n--- Training Data Date Range Per League (2020+) ---")
    
    leagues_check = sorted(df_check['league'].unique())
    
    for league in leagues_check:
        league_df = df_check[df_check['league'] == league]
        
        oldest_date = league_df['Date'].min()
        newest_date = league_df['Date'].max()
        
        print(f"\nLeague: {league.title()}")
        print(f"  Oldest match: {oldest_date.strftime('%Y-%m-%d')}")
        print(f"  Newest match: {newest_date.strftime('%Y-%m-%d')}")

except FileNotFoundError:
    print("\nCould not find 'full_processed_data.csv' to validate dates.")

In [26]:
# ==============================================================================
# SCRIPT TO VALIDATE ALL SPECIALIZED MODELS (INCLUDING XG)
# ==============================================================================
import pandas as pd
import glob
import joblib
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

try:
    # Load the master dataset that was used for training
    all_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    
    # --- Define the feature columns used for training ---
    feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
    
    # --- 1. Validate 1X2 Models ---
    print("--- Validating 1X2 Models ---")
    print(f"{'League':<20} | {'Accuracy':<10} | {'Log Loss':<10}")
    print("-" * 50)
    model_files_1x2 = sorted(glob.glob('model_1x2_*.joblib'))
    leagues_1x2 = [f.replace('model_1x2_', '').replace('.joblib', '') for f in model_files_1x2]

    for league in leagues_1x2:
        model = joblib.load(f'model_1x2_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['result_label'])
        X, y = league_data[feature_cols], league_data['result_label']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            y_pred_proba = model.predict_proba(X_test)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            loss = log_loss(y_test, y_pred_proba)
            print(f"{league.title():<20} | {acc:<10.2%} | {loss:<10.4f}")

    # --- 2. Validate Over/Under 2.5 Models ---
    print("\n--- Validating Over/Under 2.5 Models ---")
    print(f"{'League':<20} | {'Accuracy':<10} | {'Log Loss':<10}")
    print("-" * 50)
    model_files_ou = sorted(glob.glob('model_ou25_*.joblib'))
    leagues_ou = [f.replace('model_ou25_', '').replace('.joblib', '') for f in model_files_ou]

    for league in leagues_ou:
        model = joblib.load(f'model_ou25_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['over_2_5'])
        X, y = league_data[feature_cols], league_data['over_2_5']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            y_pred_proba = model.predict_proba(X_test)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            loss = log_loss(y_test, y_pred_proba)
            print(f"{league.title():<20} | {acc:<10.2%} | {loss:<10.4f}")

    # --- 3. Validate BTTS Models ---
    print("\n--- Validating BTTS (Both Teams To Score) Models ---")
    print(f"{'League':<20} | {'Accuracy':<10} | {'Log Loss':<10}")
    print("-" * 50)
    model_files_btts = sorted(glob.glob('model_btts_*.joblib'))
    leagues_btts = [f.replace('model_btts_', '').replace('.joblib', '') for f in model_files_btts]

    for league in leagues_btts:
        model = joblib.load(f'model_btts_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['btts'])
        X, y = league_data[feature_cols], league_data['btts']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            y_pred_proba = model.predict_proba(X_test)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            loss = log_loss(y_test, y_pred_proba)
            print(f"{league.title():<20} | {acc:<10.2%} | {loss:<10.4f}")
            
    # --- 4. Validate Home Expected Goals (xGH) Models ---
    print("\n--- Validating Home Expected Goals (xGH) Models ---")
    print(f"{'League':<20} | {'Mean Absolute Error':<20}")
    print("-" * 50)
    model_files_xgh = sorted(glob.glob('model_xGH_*.joblib'))
    leagues_xgh = [f.replace('model_xGH_', '').replace('.joblib', '') for f in model_files_xgh]

    for league in leagues_xgh:
        model = joblib.load(f'model_xGH_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['FTHG'])
        X, y = league_data[feature_cols], league_data['FTHG']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            y_pred = model.predict(X_test)
            mae = mean_absolute_error(y_test, y_pred)
            print(f"{league.title():<20} | {mae:<20.4f}")

    # --- 5. Validate Away Expected Goals (xGA) Models ---
    print("\n--- Validating Away Expected Goals (xGA) Models ---")
    print(f"{'League':<20} | {'Mean Absolute Error':<20}")
    print("-" * 50)
    model_files_xga = sorted(glob.glob('model_xGA_*.joblib'))
    leagues_xga = [f.replace('model_xGA_', '').replace('.joblib', '') for f in model_files_xga]

    for league in leagues_xga:
        model = joblib.load(f'model_xGA_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['FTAG'])
        X, y = league_data[feature_cols], league_data['FTAG']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            y_pred = model.predict(X_test)
            mae = mean_absolute_error(y_test, y_pred)
            print(f"{league.title():<20} | {mae:<20.4f}")

except FileNotFoundError:
    print("Could not find the 'full_processed_data.csv' or model files. Make sure they are in the correct directory.")

--- Validating 1X2 Models ---
League               | Accuracy   | Log Loss  
--------------------------------------------------
Ch                   | 53.94%     | 0.9456    
Belgian              | 61.80%     | 0.8406    
British_Champ        | 54.35%     | 0.9165    
British_Conference   | 53.03%     | 0.9745    
British_Pl           | 61.05%     | 0.8448    
Bundesliga_2         | 61.96%     | 0.8470    
Chn                  | 61.38%     | 0.7944    
Dutch                | 63.48%     | 0.7880    
Fin                  | 63.43%     | 0.8353    
French               | 63.12%     | 0.8514    
German               | 57.39%     | 0.9160    
German_2             | 52.61%     | 0.9443    
Greek                | 61.45%     | 0.8514    
Italian              | 61.05%     | 0.8784    
Japan                | 52.26%     | 0.9484    
Norsk                | 62.44%     | 0.7953    
Porto                | 76.09%     | 0.5972    
Russian              | 64.80%     | 0.7841    
Spanish              | 62.

In [19]:
# ==============================================================================
# SHOWS THE MODEL VARIABLES
# ==============================================================================

import joblib
import glob
import warnings
warnings.filterwarnings('ignore')

def display_model_params(model_path):
    """Loads a saved model and prints its key hyperparameters."""
    try:
        calibrated_model = joblib.load(model_path)
        # The actual XGBoost model is stored in the .estimator attribute
        xgb_model = calibrated_model.estimator
        params = xgb_model.get_params()
        
        # Extract the parameters we are interested in
        n_estimators = params.get('n_estimators')
        max_depth = params.get('max_depth')
        learning_rate = params.get('learning_rate')
        
        print(f"  - {model_path:<30} | n_estimators: {n_estimators}, max_depth: {max_depth}, learning_rate: {learning_rate}")
        
    except Exception as e:
        print(f"Could not read model parameters from {model_path}: {e}")

# --- 1X2 Models ---
print("--- 1X2 Model Parameters ---")
model_files_1x2 = sorted(glob.glob('model_1x2_*.joblib'))
if not model_files_1x2:
    print("  No 1X2 models found.")
for f in model_files_1x2:
    display_model_params(f)

# --- Over/Under 2.5 Models ---
print("\n--- Over/Under 2.5 Model Parameters ---")
model_files_ou = sorted(glob.glob('model_ou25_*.joblib'))
if not model_files_ou:
    print("  No O/U 2.5 models found.")
for f in model_files_ou:
    display_model_params(f)

# --- BTTS Models ---
print("\n--- BTTS Model Parameters ---")
model_files_btts = sorted(glob.glob('model_btts_*.joblib'))
if not model_files_btts:
    print("  No BTTS models found.")
for f in model_files_btts:
    display_model_params(f)

--- 1X2 Model Parameters ---
  - model_1x2_CH.joblib            | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_belgian.joblib       | n_estimators: 1109, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_champ.joblib | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_conference.joblib | n_estimators: 935, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_pl.joblib    | n_estimators: 1315, max_depth: 3, learning_rate: 0.005
  - model_1x2_bundesliga_2.joblib  | n_estimators: 875, max_depth: 3, learning_rate: 0.01
  - model_1x2_chn.joblib           | n_estimators: 1143, max_depth: 3, learning_rate: 0.005
  - model_1x2_dutch.joblib         | n_estimators: 1185, max_depth: 3, learning_rate: 0.005
  - model_1x2_fin.joblib           | n_estimators: 1275, max_depth: 3, learning_rate: 0.005
  - model_1x2_french.joblib        | n_estimators: 849, max_depth: 3, learning_rate: 0.005
  - model_1x2_german.joblib        | n_estimators: 8

In [1]:
# ==============================================================================
# CELL 2: INTERACTIVE PREDICTOR (FINAL CORRECTED LOGIC)
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from IPython.display import display, HTML
from ipywidgets import widgets, interactive_output, VBox

# --- Load Saved Models and Data ---
try:
    historical_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    all_teams = sorted(pd.concat([historical_data['HomeTeam'], historical_data['AwayTeam']]).unique())
    
    # Create a mapping of each team to its league for model loading
    team_to_league_map = historical_data.drop_duplicates(subset=['HomeTeam', 'league']).set_index('HomeTeam')['league'].to_dict()
    away_map = historical_data.drop_duplicates(subset=['AwayTeam', 'league']).set_index('AwayTeam')['league'].to_dict()
    team_to_league_map.update(away_map)
    
    print("--- Prediction Environment Ready ---")
except FileNotFoundError:
    print("--- Model files not found. Please run the training cell (Cell 1) first. ---")

# --- Create a Master Table of Final Team Stats ---
final_stats_list = []
for team_name in all_teams:
    last_match = historical_data[(historical_data['HomeTeam'] == team_name) | (historical_data['AwayTeam'] == team_name)].iloc[-1]
    
    if last_match['HomeTeam'] == team_name:
        stats = {
            'Team': team_name,
            'FinalElo': last_match['FinalHomeElo'],
            'FinalForm': last_match['HomeForm'],
            'FinalAvgGoalsScored': last_match['AvgHomeGoals'],
            'FinalAvgGoalsConceded': last_match['AvgHomeConceded']
        }
    else: # They were the away team
        stats = {
            'Team': team_name,
            'FinalElo': last_match['FinalAwayElo'],
            'FinalForm': last_match['AwayForm'],
            'FinalAvgGoalsScored': last_match['AvgAwayGoals'],
            'FinalAvgGoalsConceded': last_match['AvgAwayConceded']
        }
    final_stats_list.append(stats)

final_team_stats = pd.DataFrame(final_stats_list).set_index('Team')
print("--- Master stats table created successfully. ---")


# --- Define the Prediction Function (with CORRECTED logic) ---
def predict_live_match(home_team, away_team):
    if not home_team or not away_team or home_team == away_team:
        return 
    
    try:
        league = team_to_league_map.get(home_team)
        if not league:
            display(HTML(f"<p style='color:red;'>Could not determine league for {home_team}.</p>"))
            return
            
        model_1x2 = joblib.load(f'model_1x2_{league}.joblib')
        model_ou25 = joblib.load(f'model_ou25_{league}.joblib')
        model_btts = joblib.load(f'model_btts_{league}.joblib')
        
        home_stats = final_team_stats.loc[home_team]
        away_stats = final_team_stats.loc[away_team]
    except (IndexError, FileNotFoundError):
        display(HTML(f"<p style='color:red;'>Could not load models or find stats. Ensure models for '{league}' were trained.</p>"))
        return
        
    # --- THIS IS THE CORRECTED FEATURE LOGIC ---
    fixture = {
        'Elo_diff': home_stats['FinalElo'] - away_stats['FinalElo'], 
        'HomeForm': home_stats['FinalForm'],
        'AwayForm': away_stats['FinalForm'], 
        'AvgHomeGoals': home_stats['FinalAvgGoalsScored'], 
        'AvgAwayGoals': away_stats['FinalAvgGoalsScored'],      # Corrected
        'AvgHomeConceded': home_stats['FinalAvgGoalsConceded'], 
        'AvgAwayConceded': away_stats['FinalAvgGoalsConceded']   # Corrected
    }
    fixture_df = pd.DataFrame([fixture])
    feature_cols = list(fixture.keys())

    # Generate predictions
    probs_1x2 = model_1x2.predict_proba(fixture_df[feature_cols])[0]
    probs_ou = model_ou25.predict_proba(fixture_df[feature_cols])[0]
    probs_btts = model_btts.predict_proba(fixture_df[feature_cols])[0]
        
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 10px; margin-top: 10px;">
    <h3>Prediction: {home_team} vs {away_team}</h3>
    <p><i>(Using specialized models for the <b>{league.title()}</b> league)</i></p>
    <table border="1" style="width:100%; text-align:center;">
      <tr style="background-color:#f2f2f2;"><th>1X2 Market</th><th>Over/Under 2.5</th><th>BTTS</th></tr>
      <tr>
        <td><b>Home:</b> {probs_1x2[2]:.1%} | <b>Draw:</b> {probs_1x2[1]:.1%} | <b>Away:</b> {probs_1x2[0]:.1%}</td>
        <td><b>Over:</b> {probs_ou[1]:.1%} | <b>Under:</b> {probs_ou[0]:.1%}</td>
        <td><b>Yes:</b> {probs_btts[1]:.1%} | <b>No:</b> {probs_btts[0]:.1%}</td>
      </tr>
    </table></div>"""))

# --- Create and Display the Interactive Dropdown Widgets ---
style = {'description_width': 'initial'}
home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:', style=style, layout={'width': '500px'})
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:', style=style, layout={'width': '500px'}, value=all_teams[1] if len(all_teams) > 1 else None)

ui = VBox([home_dropdown, away_dropdown])
out = widgets.interactive_output(predict_live_match, {'home_team': home_dropdown, 'away_team': away_dropdown})

display(ui, out)

--- Prediction Environment Ready ---
--- Master stats table created successfully. ---


Output()

In [ ]:
# ==============================================================================
# FINAL INTERACTIVE PREDICTOR (WITH EXACT SCORE PREDICTION)
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from IPython.display import display, HTML
from ipywidgets import widgets, interactive_output, VBox
from scipy.stats import poisson # We need this for the Poisson calculation
import warnings
warnings.filterwarnings('ignore')

# --- Load Data and Create Team-to-League Map ---
try:
    historical_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    all_teams = sorted(pd.concat([historical_data['HomeTeam'], historical_data['AwayTeam']]).unique())
    team_to_league_map = historical_data.drop_duplicates(subset=['HomeTeam', 'league']).set_index('HomeTeam')['league'].to_dict()
    away_map = historical_data.drop_duplicates(subset=['AwayTeam', 'league']).set_index('AwayTeam')['league'].to_dict()
    team_to_league_map.update(away_map)
    print("--- Prediction Environment Ready ---")
except FileNotFoundError:
    print("--- Model files not found. Please run the training cell(s) first. ---")

# --- Create a Master Table of Final Team Stats ---
final_stats_list = []
for team_name in all_teams:
    overall_matches = historical_data[(historical_data['HomeTeam'] == team_name) | (historical_data['AwayTeam'] == team_name)]
    if overall_matches.empty: continue
    last_match_overall = overall_matches.iloc[-1]
    
    if last_match_overall['HomeTeam'] == team_name:
        elo, form = last_match_overall['FinalHomeElo'], last_match_overall['HomeForm']
    else:
        elo, form = last_match_overall['FinalAwayElo'], last_match_overall['AwayForm']
    
    home_matches = historical_data[historical_data['HomeTeam'] == team_name]
    avg_home_goals_scored = home_matches.iloc[-1]['AvgHomeGoals'] if not home_matches.empty else 0
    avg_home_goals_conceded = home_matches.iloc[-1]['AvgHomeConceded'] if not home_matches.empty else 0
    
    away_matches = historical_data[historical_data['AwayTeam'] == team_name]
    avg_away_goals_scored = away_matches.iloc[-1]['AvgAwayGoals'] if not away_matches.empty else 0
    avg_away_goals_conceded = away_matches.iloc[-1]['AvgAwayConceded'] if not away_matches.empty else 0
        
    stats = {'Team': team_name, 'FinalElo': elo, 'FinalForm': form,
             'FinalAvgHomeGoalsScored': avg_home_goals_scored, 'FinalAvgHomeGoalsConceded': avg_home_goals_conceded,
             'FinalAvgAwayGoalsScored': avg_away_goals_scored, 'FinalAvgAwayGoalsConceded': avg_away_goals_conceded}
    final_stats_list.append(stats)

final_team_stats = pd.DataFrame(final_stats_list).set_index('Team')
print("--- Master stats table created successfully. ---")

# --- Define the Prediction Function ---
def predict_live_match(home_team, away_team):
    if not home_team or not away_team or home_team == away_team: return 
    
    try:
        league = team_to_league_map.get(home_team)
        if not league:
            display(HTML(f"<p style='color:red;'>Could not determine league for {home_team}.</p>"))
            return
            
        # Load all FIVE specialized models for the league
        model_1x2 = joblib.load(f'model_1x2_{league}.joblib')
        model_ou25 = joblib.load(f'model_ou25_{league}.joblib')
        model_btts = joblib.load(f'model_btts_{league}.joblib')
        model_xgh = joblib.load(f'model_xGH_{league}.joblib')
        model_xga = joblib.load(f'model_xGA_{league}.joblib')
        
        home_stats = final_team_stats.loc[home_team]
        away_stats = final_team_stats.loc[away_team]
    except (IndexError, FileNotFoundError, KeyError):
        display(HTML(f"<p style='color:red;'>Could not load models or find stats. Ensure models for '{league}' were trained.</p>"))
        return
        
    fixture = {
        'Elo_diff': home_stats['FinalElo'] - away_stats['FinalElo'], 'HomeForm': home_stats['FinalForm'],
        'AwayForm': away_stats['FinalForm'], 'AvgHomeGoals': home_stats['FinalAvgHomeGoalsScored'], 
        'AvgAwayGoals': away_stats['FinalAvgAwayGoalsScored'], 'AvgHomeConceded': home_stats['FinalAvgHomeGoalsConceded'], 
        'AvgAwayConceded': away_stats['FinalAvgAwayGoalsConceded']
    }
    fixture_df = pd.DataFrame([fixture])
    feature_cols = list(fixture.keys())

    # --- Generate predictions from ALL models ---
    probs_1x2 = model_1x2.predict_proba(fixture_df[feature_cols])[0]
    probs_ou = model_ou25.predict_proba(fixture_df[feature_cols])[0]
    probs_btts = model_btts.predict_proba(fixture_df[feature_cols])[0]
    
    # Predict expected goals
    home_xg = model_xgh.predict(fixture_df[feature_cols])[0]
    away_xg = model_xga.predict(fixture_df[feature_cols])[0]
    
    # --- Calculate Exact Score Probabilities ---
    score_probs = []
    for i in range(6): # Home goals 0-5
        for j in range(6): # Away goals 0-5
            prob = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            score_probs.append({'score': f'{i}-{j}', 'prob': prob})
    
    # Get the top 6 most likely scores
    top_6_scores = sorted(score_probs, key=lambda x: x['prob'], reverse=True)[:6]
    top_scores_str = " | ".join([f"{s['score']} ({s['prob']:.1%})" for s in top_6_scores])
        
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 10px; margin-top: 10px;">
    <h3>Prediction: {home_team} vs {away_team}</h3>
    <p><i>(Using specialized models for the <b>{league.title()}</b> league)</i></p>
    <table border="1" style="width:100%; text-align:center;">
      <tr style="background-color:#f2f2f2;"><th>1X2 Market</th><th>Over/Under 2.5</th><th>BTTS</th></tr>
      <tr>
        <td><b>Home:</b> {probs_1x2[2]:.1%} | <b>Draw:</b> {probs_1x2[1]:.1%} | <b>Away:</b> {probs_1x2[0]:.1%}</td>
        <td><b>Over:</b> {probs_ou[1]:.1%} | <b>Under:</b> {probs_ou[0]:.1%}</td>
        <td><b>Yes:</b> {probs_btts[1]:.1%} | <b>No:</b> {probs_btts[0]:.1%}</td>
      </tr>
      <tr style="background-color:#f2f2f2;"><th colspan="3">Most Likely Exact Scores (xG H: {home_xg:.2f}, A: {away_xg:.2f})</th></tr>
      <tr><td colspan="3"><b>{top_scores_str}</b></td></tr>
    </table></div>"""))

# --- Create and Display the Interactive Dropdown Widgets ---
style = {'description_width': 'initial'}
home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:', style=style, layout={'width': '500px'})
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:', style=style, layout={'width': '500px'}, value=all_teams[1] if len(all_teams) > 1 else None)

ui = VBox([home_dropdown, away_dropdown])
out = widgets.interactive_output(predict_live_match, {'home_team': home_dropdown, 'away_team': away_dropdown})

display(ui, out)

--- Prediction Environment Ready ---
--- Master stats table created successfully. ---


Output()

In [ ]:
# ==============================================================================
# FINAL INTERACTIVE PREDICTOR (WITH XG-DERIVED SIDE NOTES)
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from IPython.display import display, HTML
from ipywidgets import widgets, interactive_output, VBox
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

# --- Load Data and Create Team-to-League Map ---
try:
    historical_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    all_teams = sorted(pd.concat([historical_data['HomeTeam'], historical_data['AwayTeam']]).unique())
    team_to_league_map = historical_data.drop_duplicates(subset=['HomeTeam', 'league']).set_index('HomeTeam')['league'].to_dict()
    away_map = historical_data.drop_duplicates(subset=['AwayTeam', 'league']).set_index('AwayTeam')['league'].to_dict()
    team_to_league_map.update(away_map)
    print("--- Prediction Environment Ready ---")
except FileNotFoundError:
    print("--- Model files not found. Please run the training cell(s) first. ---")

# --- Create a Master Table of Final Team Stats ---
final_stats_list = []
for team_name in all_teams:
    overall_matches = historical_data[(historical_data['HomeTeam'] == team_name) | (historical_data['AwayTeam'] == team_name)]
    if overall_matches.empty: continue
    last_match_overall = overall_matches.iloc[-1]
    
    if last_match_overall['HomeTeam'] == team_name:
        elo, form = last_match_overall['FinalHomeElo'], last_match_overall['HomeForm']
    else:
        elo, form = last_match_overall['FinalAwayElo'], last_match_overall['AwayForm']
    
    home_matches = historical_data[historical_data['HomeTeam'] == team_name]
    avg_home_goals_scored = home_matches.iloc[-1]['AvgHomeGoals'] if not home_matches.empty else 0
    avg_home_goals_conceded = home_matches.iloc[-1]['AvgHomeConceded'] if not home_matches.empty else 0
    
    away_matches = historical_data[historical_data['AwayTeam'] == team_name]
    avg_away_goals_scored = away_matches.iloc[-1]['AvgAwayGoals'] if not away_matches.empty else 0
    avg_away_goals_conceded = away_matches.iloc[-1]['AvgAwayConceded'] if not away_matches.empty else 0
        
    stats = {'Team': team_name, 'FinalElo': elo, 'FinalForm': form,
             'FinalAvgHomeGoalsScored': avg_home_goals_scored, 'FinalAvgHomeGoalsConceded': avg_home_goals_conceded,
             'FinalAvgAwayGoalsScored': avg_away_goals_scored, 'FinalAvgAwayGoalsConceded': avg_away_goals_conceded}
    final_stats_list.append(stats)
final_team_stats = pd.DataFrame(final_stats_list).set_index('Team')
print("--- Master stats table created successfully. ---")

# --- Define the Prediction Function ---
def predict_live_match(home_team, away_team):
    if not home_team or not away_team or home_team == away_team: return 
    
    try:
        league = team_to_league_map.get(home_team)
        if not league:
            display(HTML(f"<p style='color:red;'>Could not determine league for {home_team}.</p>"))
            return
            
        # Load all models for the league
        model_1x2 = joblib.load(f'model_1x2_{league}.joblib')
        model_ou25 = joblib.load(f'model_ou25_{league}.joblib')
        model_btts = joblib.load(f'model_btts_{league}.joblib')
        model_xgh = joblib.load(f'model_xGH_{league}.joblib')
        model_xga = joblib.load(f'model_xGA_{league}.joblib')
        
        home_stats = final_team_stats.loc[home_team]
        away_stats = final_team_stats.loc[away_team]
    except (IndexError, FileNotFoundError, KeyError):
        display(HTML(f"<p style='color:red;'>Could not load models or find stats. Ensure models for '{league}' were trained.</p>"))
        return
        
    fixture = {'Elo_diff': home_stats['FinalElo'] - away_stats['FinalElo'], 'HomeForm': home_stats['FinalForm'],
               'AwayForm': away_stats['FinalForm'], 'AvgHomeGoals': home_stats['FinalAvgHomeGoalsScored'], 
               'AvgAwayGoals': away_stats['FinalAvgAwayGoalsScored'], 'AvgHomeConceded': home_stats['FinalAvgHomeGoalsConceded'], 
               'AvgAwayConceded': away_stats['FinalAvgAwayGoalsConceded']}
    fixture_df = pd.DataFrame([fixture])
    feature_cols = list(fixture.keys())

    # --- Primary Predictions (from specialized models) ---
    probs_1x2 = model_1x2.predict_proba(fixture_df[feature_cols])[0]
    probs_ou = model_ou25.predict_proba(fixture_df[feature_cols])[0]
    probs_btts = model_btts.predict_proba(fixture_df[feature_cols])[0]
    
    # --- Secondary Predictions (derived from xG models) ---
    home_xg = model_xgh.predict(fixture_df[feature_cols])[0]
    away_xg = model_xga.predict(fixture_df[feature_cols])[0]
    
    # Calculate score probabilities
    score_probs = {}
    for i in range(7): # Home goals 0-6
        for j in range(7): # Away goals 0-6
            prob = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            score_probs[f'{i}-{j}'] = prob
            
    # Derive O/U and BTTS from score probabilities
    prob_over_25_xg = sum(prob for score, prob in score_probs.items() if int(score.split('-')[0]) + int(score.split('-')[1]) > 2)
    prob_btts_yes_xg = sum(prob for score, prob in score_probs.items() if int(score.split('-')[0]) > 0 and int(score.split('-')[1]) > 0)
    
    # Get top 3 most likely scores
    top_3_scores = sorted(score_probs.items(), key=lambda item: item[1], reverse=True)[:3]
    top_scores_str = " | ".join([f"{s[0]} ({s[1]:.1%})" for s in top_3_scores])
        
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 10px; margin-top: 10px;">
    <h3>Prediction: {home_team} vs {away_team}</h3>
    <p><i>(Using specialized models for the <b>{league.title()}</b> league)</i></p>
    <table border="1" style="width:100%; text-align:center;">
      <tr style="background-color:#f2f2f2;"><th>1X2 Market</th><th>Over/Under 2.5</th><th>BTTS</th></tr>
      <tr>
        <td><b>Home:</b> {probs_1x2[2]:.1%} | <b>Draw:</b> {probs_1x2[1]:.1%} | <b>Away:</b> {probs_1x2[0]:.1%}</td>
        <td><b>Over:</b> {probs_ou[1]:.1%} | <b>Under:</b> {probs_ou[0]:.1%}</td>
        <td><b>Yes:</b> {probs_btts[1]:.1%} | <b>No:</b> {probs_btts[0]:.1%}</td>
      </tr>
      <tr style="background-color:#f9f9f9; font-style: italic;"><td colspan="3"><b>Side Note: Predictions Derived from Expected Goals (H: {home_xg:.2f}, A: {away_xg:.2f})</b></td></tr>
      <tr style="font-style: italic;">
        <td><b>Top Scores:</b> {top_scores_str}</td>
        <td><b>Over:</b> {prob_over_25_xg:.1%} | <b>Under:</b> {1-prob_over_25_xg:.1%}</td>
        <td><b>Yes:</b> {prob_btts_yes_xg:.1%} | <b>No:</b> {1-prob_btts_yes_xg:.1%}</td>
      </tr>
    </table></div>"""))

# --- Create and Display the Interactive Dropdown Widgets ---
style = {'description_width': 'initial'}
home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:', style=style, layout={'width': '500px'})
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:', style=style, layout={'width': '500px'}, value=all_teams[1] if len(all_teams) > 1 else None)
ui = VBox([home_dropdown, away_dropdown])
out = widgets.interactive_output(predict_live_match, {'home_team': home_dropdown, 'away_team': away_dropdown})
display(ui, out)

--- Prediction Environment Ready ---
--- Master stats table created successfully. ---


Output()

In [ ]:
# ==============================================================================
# CELL FOR AUTOMATED INCREMENTAL DATA UPDATE
# ==============================================================================
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# --- PASTE ALL HELPER FUNCTIONS FROM YOUR TRAINING SCRIPT HERE ---
# (This part is the same as before - just copy/paste the functions
# like calculate_elo_for_league, create_features, etc.)

def expected_outcome(rating1, rating2): return 1 / (1 + 10 ** ((rating2 - rating1) / 400))
def update_elo(home_elo, away_elo, fthg, ftag, k=20):
    e_home = expected_outcome(home_elo, away_elo)
    s_home = 1 if fthg > ftag else 0 if ftag > fthg else 0.5
    updated_home_elo = home_elo + k * (s_home - e_home)
    updated_away_elo = away_elo - (updated_home_elo - home_elo)
    return updated_home_elo, updated_away_elo
def calculate_elo_for_league(df, initial_elo=1500, k=20):
    elo_ratings = {}
    home_elos, away_elos = [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        home_elo, away_elo = elo_ratings.get(home_team, initial_elo), elo_ratings.get(away_team, initial_elo)
        home_elos.append(home_elo); away_elos.append(away_elo)
        new_home_elo, new_away_elo = update_elo(home_elo, away_elo, row['FTHG'], row['FTAG'], k)
        elo_ratings[home_team], elo_ratings[away_team] = new_home_elo, new_away_elo
    df['HomeElo'], df['AwayElo'] = home_elos, away_elos
    df['FinalHomeElo'] = df['HomeTeam'].map(elo_ratings)
    df['FinalAwayElo'] = df['AwayTeam'].map(elo_ratings)
    return df
def get_form_points(df, n_matches=5):
    team_matches = {}
    for _, row in df.iterrows():
        res = row['FTR']
        if res == 'H': home_pts, away_pts = 3, 0
        elif res == 'A': home_pts, away_pts = 0, 3
        else: home_pts, away_pts = 1, 1
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_matches: team_matches[home_team] = []
        if away_team not in team_matches: team_matches[away_team] = []
        team_matches[home_team].append(home_pts)
        team_matches[away_team].append(away_pts)
    home_form, away_form = [], []
    for _, row in df.iterrows():
        home_form.append(sum(team_matches[row['HomeTeam']][-n_matches-1:-1]))
        away_form.append(sum(team_matches[row['AwayTeam']][-n_matches-1:-1]))
    df['HomeForm'], df['AwayForm'] = home_form, away_form
    return df
def get_goal_averages(df):
    team_stats = {}
    home_goals_avg, away_goals_avg, home_conceded_avg, away_conceded_avg = [], [], [], []
    for _, row in df.iterrows():
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        if home_team not in team_stats: team_stats[home_team] = {'scored': [], 'conceded': []}
        if away_team not in team_stats: team_stats[away_team] = {'scored': [], 'conceded': []}
        home_goals_avg.append(np.mean(team_stats[home_team]['scored']) if team_stats[home_team]['scored'] else 0)
        home_conceded_avg.append(np.mean(team_stats[home_team]['conceded']) if team_stats[home_team]['conceded'] else 0)
        away_goals_avg.append(np.mean(team_stats[away_team]['scored']) if team_stats[away_team]['scored'] else 0)
        away_conceded_avg.append(np.mean(team_stats[away_team]['conceded']) if team_stats[away_team]['conceded'] else 0)
        team_stats[home_team]['scored'].append(row['FTHG']); team_stats[home_team]['conceded'].append(row['FTAG'])
        team_stats[away_team]['scored'].append(row['FTAG']); team_stats[away_team]['conceded'].append(row['FTHG'])
    df['AvgHomeGoals'], df['AvgHomeConceded'] = home_goals_avg, home_conceded_avg
    df['AvgAwayGoals'], df['AvgAwayConceded'] = away_goals_avg, away_conceded_avg
    return df
def create_features(df):
    df['result_label'] = df['FTR'].map({'H': 2, 'D': 1, 'A': 0})
    df['over_2_5'] = ((df['FTHG'] + df['FTAG']) > 2.5).astype(int)
    df['btts'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    df['Elo_diff'] = df['HomeElo'] - df['AwayElo']
    df = get_form_points(df)
    df = get_goal_averages(df)
    df = df.dropna(subset=['result_label'])
    df['result_label'] = df['result_label'].astype(int)
    return df

# --- Main Update Logic ---
try:
    # 1. Load existing processed data
    existing_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    print(f"Loaded {len(existing_data)} already-processed historical matches.")

    # 2. Scan the /data folder and load ALL raw files
    # This logic is copied from your original training script for consistency
    all_files = glob.glob('data/**/*.csv', recursive=True)
    raw_dfs = []
    print(f"Scanning {len(all_files)} files in the data directory...")
    for f in all_files:
        league_name = os.path.basename(os.path.dirname(f))
        # Skip files where league_name is invalid (e.g., 'data' or other non-league folders)
        if league_name in ['data', 'global', 'DANSK']:
            print(f"  Skipping file in invalid folder: {f}")
            continue
        delimiter = ';' if any(x in f for x in ['CHN', 'JPN', 'CH/']) else ','
        temp_df = pd.read_csv(f, delimiter=delimiter, on_bad_lines='skip', encoding='latin1')
        temp_df['league'] = league_name
        temp_df = temp_df.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam', 'HG': 'FTHG', 'AG': 'FTAG', 'Res': 'FTR'})
        raw_dfs.append(temp_df)
    all_raw_data = pd.concat(raw_dfs, ignore_index=True)
    all_raw_data['Date'] = pd.to_datetime(all_raw_data['Date'], dayfirst=True, errors='coerce')
    all_raw_data.dropna(subset=['Date', 'HomeTeam', 'AwayTeam'], inplace=True)

    # 3. Identify ONLY the new matches
    # Create a unique ID for each match to allow for a reliable comparison
    existing_data['match_id'] = existing_data['Date'].dt.strftime('%Y-%m-%d') + '_' + existing_data['HomeTeam'] + '_' + existing_data['AwayTeam']
    all_raw_data['match_id'] = all_raw_data['Date'].dt.strftime('%Y-%m-%d') + '_' + all_raw_data['HomeTeam'] + '_' + all_raw_data['AwayTeam']
    
    # Filter the raw data to find rows where the match_id is NOT in the existing data
    new_matches = all_raw_data[~all_raw_data['match_id'].isin(existing_data['match_id'])].copy()
    
    if new_matches.empty:
        print("\n✅ No new matches found. Your data is already up to date.")
    else:
        print(f"Found {len(new_matches)} new matches to integrate.")
        
        # 4. Combine old and new data for processing
        common_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'league']
        combined_data = pd.concat([
            existing_data[common_cols],
            new_matches[common_cols]
        ], ignore_index=True)
        
        # 5. Sort by date to ensure chronological processing
        combined_data = combined_data.sort_values('Date').reset_index(drop=True)
        
        # 6. Re-calculate all features on the full, combined dataset
        print("Re-calculating features chronologically (Elo, Form, Averages)...")
        updated_full_data = []
        for league in combined_data['league'].unique():
            league_df = combined_data[combined_data['league'] == league].copy()
            df_with_elo = calculate_elo_for_league(league_df)
            df_featured = create_features(df_with_elo)
            updated_full_data.append(df_featured)
        
        final_df = pd.concat(updated_full_data, ignore_index=True).sort_values('Date').reset_index(drop=True)

        # 7. Overwrite the old file with the newly updated data
        final_df.to_csv('full_processed_data.csv', index=False)
        print(f"\n✅ SUCCESS: Data updated. 'full_processed_data.csv' now contains {len(final_df)} matches.")
        print("You can now re-run the 'FINAL INTERACTIVE PREDICTOR' cell.")

except FileNotFoundError:
    print("❌ ERROR: Could not find 'full_processed_data.csv'. Run the full training script once first.")
except Exception as e:
    print(f"An error occurred: {e}")

Loaded 50039 already-processed historical matches.
Scanning 90 files in the data directory...
  Skipping file in invalid folder: data/DANSK/DNK.csv
  Skipping file in invalid folder: data/global/international.csv
Found 166 new matches to integrate.
Re-calculating features chronologically (Elo, Form, Averages)...

✅ SUCCESS: Data updated. 'full_processed_data.csv' now contains 50205 matches.
You can now re-run the 'FINAL INTERACTIVE PREDICTOR' cell.


In [3]:
# ==============================================================================
# FINAL INTERACTIVE PREDICTOR (WITH XG-DERIVED SIDE NOTES)
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from IPython.display import display, HTML
from ipywidgets import widgets, interactive_output, VBox
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

# --- Load Data and Create Team-to-League Map ---
try:
    historical_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    all_teams = sorted(pd.concat([historical_data['HomeTeam'], historical_data['AwayTeam']]).unique())
    team_to_league_map = historical_data.drop_duplicates(subset=['HomeTeam', 'league']).set_index('HomeTeam')['league'].to_dict()
    away_map = historical_data.drop_duplicates(subset=['AwayTeam', 'league']).set_index('AwayTeam')['league'].to_dict()
    team_to_league_map.update(away_map)
    print("--- Prediction Environment Ready ---")
except FileNotFoundError:
    print("--- Model files not found. Please run the training cell(s) first. ---")

# --- Create a Master Table of Final Team Stats ---
final_stats_list = []
for team_name in all_teams:
    overall_matches = historical_data[(historical_data['HomeTeam'] == team_name) | (historical_data['AwayTeam'] == team_name)]
    if overall_matches.empty: continue
    last_match_overall = overall_matches.iloc[-1]
    
    if last_match_overall['HomeTeam'] == team_name:
        elo, form = last_match_overall['FinalHomeElo'], last_match_overall['HomeForm']
    else:
        elo, form = last_match_overall['FinalAwayElo'], last_match_overall['AwayForm']
    
    home_matches = historical_data[historical_data['HomeTeam'] == team_name]
    avg_home_goals_scored = home_matches.iloc[-1]['AvgHomeGoals'] if not home_matches.empty else 0
    avg_home_goals_conceded = home_matches.iloc[-1]['AvgHomeConceded'] if not home_matches.empty else 0
    
    away_matches = historical_data[historical_data['AwayTeam'] == team_name]
    avg_away_goals_scored = away_matches.iloc[-1]['AvgAwayGoals'] if not away_matches.empty else 0
    avg_away_goals_conceded = away_matches.iloc[-1]['AvgAwayConceded'] if not away_matches.empty else 0
        
    stats = {'Team': team_name, 'FinalElo': elo, 'FinalForm': form,
             'FinalAvgHomeGoalsScored': avg_home_goals_scored, 'FinalAvgHomeGoalsConceded': avg_home_goals_conceded,
             'FinalAvgAwayGoalsScored': avg_away_goals_scored, 'FinalAvgAwayGoalsConceded': avg_away_goals_conceded}
    final_stats_list.append(stats)
final_team_stats = pd.DataFrame(final_stats_list).set_index('Team')
print("--- Master stats table created successfully. ---")

# --- Define the Prediction Function ---
# --- Define the Prediction Function (UPDATED) ---
def predict_live_match(home_team, away_team):
    if not home_team or not away_team or home_team == away_team: return 
    
    try:
        league = team_to_league_map.get(home_team)
        if not league:
            display(HTML(f"<p style='color:red;'>Could not determine league for {home_team}.</p>"))
            return
            
        # Load all models for the league
        model_1x2 = joblib.load(f'model_1x2_{league}.joblib')
        model_ou25 = joblib.load(f'model_ou25_{league}.joblib')
        model_btts = joblib.load(f'model_btts_{league}.joblib')
        model_xgh = joblib.load(f'model_xGH_{league}.joblib')
        model_xga = joblib.load(f'model_xGA_{league}.joblib')
        
        home_stats = final_team_stats.loc[home_team]
        away_stats = final_team_stats.loc[away_team]
    except (IndexError, FileNotFoundError, KeyError):
        display(HTML(f"<p style='color:red;'>Could not load models or find stats. Ensure models for '{league}' were trained.</p>"))
        return
        
    fixture = {'Elo_diff': home_stats['FinalElo'] - away_stats['FinalElo'], 'HomeForm': home_stats['FinalForm'],
               'AwayForm': away_stats['FinalForm'], 'AvgHomeGoals': home_stats['FinalAvgHomeGoalsScored'], 
               'AvgAwayGoals': away_stats['FinalAvgAwayGoalsScored'], 'AvgHomeConceded': home_stats['FinalAvgHomeGoalsConceded'], 
               'AvgAwayConceded': away_stats['FinalAvgAwayGoalsConceded']}
    fixture_df = pd.DataFrame([fixture])
    feature_cols = list(fixture.keys())

    # --- Primary Predictions (from specialized models) ---
    probs_1x2 = model_1x2.predict_proba(fixture_df[feature_cols])[0]
    probs_ou = model_ou25.predict_proba(fixture_df[feature_cols])[0]
    probs_btts = model_btts.predict_proba(fixture_df[feature_cols])[0]
    
    # --- NEW: Calculate odds and double chance ---
    def safe_odds(p):
        return 1 / p if p > 0 else float('inf')

    # 1X2 probabilities and odds
    prob_A, prob_D, prob_H = probs_1x2[0], probs_1x2[1], probs_1x2[2]
    odds_A, odds_D, odds_H = safe_odds(prob_A), safe_odds(prob_D), safe_odds(prob_H)

    # Double Chance probabilities and odds
    prob_1X = prob_H + prob_D
    prob_X2 = prob_D + prob_A
    prob_12 = prob_H + prob_A
    odds_1X, odds_X2, odds_12 = safe_odds(prob_1X), safe_odds(prob_X2), safe_odds(prob_12)

    # O/U 2.5 probabilities and odds
    prob_U25, prob_O25 = probs_ou[0], probs_ou[1]
    odds_U25, odds_O25 = safe_odds(prob_U25), safe_odds(prob_O25)
    
    # BTTS probabilities and odds
    prob_btts_no, prob_btts_yes = probs_btts[0], probs_btts[1]
    odds_btts_no, odds_btts_yes = safe_odds(prob_btts_no), safe_odds(prob_btts_yes)

    # --- Secondary Predictions (derived from xG models) ---
    home_xg = model_xgh.predict(fixture_df[feature_cols])[0]
    away_xg = model_xga.predict(fixture_df[feature_cols])[0]
    
    score_probs = {f'{i}-{j}': poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg) for i in range(7) for j in range(7)}
    top_6_scores = sorted(score_probs.items(), key=lambda item: item[1], reverse=True)[:6]
    top_scores_str = " | ".join([f"{s[0]} ({s[1]:.1%})" for s in top_6_scores])

    # --- Display the updated UI ---
    display(HTML(f"""
    <div style="font-family: Arial, sans-serif; border: 1px solid #ccc; padding: 12px; margin-top: 10px; border-radius: 5px;">
    <h3>{home_team} vs {away_team}</h3>
    <p style="margin-bottom: 15px;"><i>Prediction based on <b>{league.replace('_', ' ').title()}</b> league model</i></p>
    
    <table style="width:100%; border-collapse: collapse; text-align:center;">
      <tr style="background-color:#f2f2f2;">
        <th style="padding: 8px; border-bottom: 1px solid #ddd;">Market</th>
        <th style="padding: 8px; border-bottom: 1px solid #ddd;">Prediction & (Calculated Odds)</th>
      </tr>
      <tr>
        <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">1X2</td>
        <td style="padding: 8px; border-bottom: 1px solid #ddd;">
            <b>H:</b> {prob_H:.1%} (<b style="color:#0056b3;">{odds_H:.2f}</b>) | 
            <b>D:</b> {prob_D:.1%} (<b style="color:#0056b3;">{odds_D:.2f}</b>) | 
            <b>A:</b> {prob_A:.1%} (<b style="color:#0056b3;">{odds_A:.2f}</b>)
        </td>
      </tr>
      <tr>
        <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">Double Chance</td>
        <td style="padding: 8px; border-bottom: 1px solid #ddd;">
            <b>1X:</b> {prob_1X:.1%} (<b style="color:#0056b3;">{odds_1X:.2f}</b>) | 
            <b>X2:</b> {prob_X2:.1%} (<b style="color:#0056b3;">{odds_X2:.2f}</b>) | 
            <b>12:</b> {prob_12:.1%} (<b style="color:#0056b3;">{odds_12:.2f}</b>)
        </td>
      </tr>
      <tr>
        <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">Over/Under 2.5</td>
        <td style="padding: 8px; border-bottom: 1px solid #ddd;">
            <b>Over:</b> {prob_O25:.1%} (<b style="color:#0056b3;">{odds_O25:.2f}</b>) | 
            <b>Under:</b> {prob_U25:.1%} (<b style="color:#0056b3;">{odds_U25:.2f}</b>)
        </td>
      </tr>
      <tr>
        <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">BTTS</td>
        <td style="padding: 8px; border-bottom: 1px solid #ddd;">
            <b>Yes:</b> {prob_btts_yes:.1%} (<b style="color:#0056b3;">{odds_btts_yes:.2f}</b>) | 
            <b>No:</b> {prob_btts_no:.1%} (<b style="color:#0056b3;">{odds_btts_no:.2f}</b>)
        </td>
      </tr>
      <tr style="background-color:#f9f9f9; font-style: italic;">
          <th colspan="2" style="padding: 8px; border-bottom: 1px solid #ddd;">xG Derived Side-Notes (H: {home_xg:.2f}, A: {away_xg:.2f})</th>
      </tr>
      <tr style="font-style: italic;">
          <td colspan="2" style="padding: 8px;"><b>Most Likely Scores:</b> {top_scores_str}</td>
      </tr>
    </table>
    </div>
    """))
   

# --- Create and Display the Interactive Dropdown Widgets ---
style = {'description_width': 'initial'}
home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:', style=style, layout={'width': '500px'})
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:', style=style, layout={'width': '500px'}, value=all_teams[1] if len(all_teams) > 1 else None)
ui = VBox([home_dropdown, away_dropdown])
out = widgets.interactive_output(predict_live_match, {'home_team': home_dropdown, 'away_team': away_dropdown})
display(ui, out)

--- Prediction Environment Ready ---
--- Master stats table created successfully. ---


Output()

In [3]:
# ==============================================================================
# FINAL TRAINING SCRIPT: BUILDING THE META-MODEL (CORRECTED & COMPLETE)
# ==============================================================================
import pandas as pd
import glob
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

# --- Load and prepare the data one last time ---
try:
    all_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    print("Successfully loaded processed data.")
except FileNotFoundError:
    print("ERROR: `full_processed_data.csv` not found. Please run a previous training script first.")

# --- COMPLETE Dictionary of Best Parameters Found During Tuning ---
best_params_1x2 = {
    'CH': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005},
    'belgian': {'n_estimators': 1109, 'max_depth': 3, 'learning_rate': 0.005},
    'british_champ': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005},
    'british_conference': {'n_estimators': 935, 'max_depth': 3, 'learning_rate': 0.005},
    'british_pl': {'n_estimators': 1315, 'max_depth': 3, 'learning_rate': 0.005},
    'chn': {'n_estimators': 1143, 'max_depth': 3, 'learning_rate': 0.005},
    'dutch': {'n_estimators': 1185, 'max_depth': 3, 'learning_rate': 0.005},
    'fin': {'n_estimators': 1275, 'max_depth': 3, 'learning_rate': 0.005},
    'french': {'n_estimators': 849, 'max_depth': 3, 'learning_rate': 0.005},
    'german': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005},
    'german_2': {'n_estimators': 831, 'max_depth': 3, 'learning_rate': 0.005},
    'greek': {'n_estimators': 803, 'max_depth': 3, 'learning_rate': 0.005},
    'italian': {'n_estimators': 809, 'max_depth': 3, 'learning_rate': 0.005},
    'japan': {'n_estimators': 849, 'max_depth': 3, 'learning_rate': 0.005},
    'norsk': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005},
    'porto': {'n_estimators': 1019, 'max_depth': 3, 'learning_rate': 0.01},
    'russian': {'n_estimators': 879, 'max_depth': 3, 'learning_rate': 0.005},
    'spanish': {'n_estimators': 879, 'max_depth': 3, 'learning_rate': 0.005},
    'spanish_2': {'n_estimators': 1811, 'max_depth': 3, 'learning_rate': 0.005},
    'swedish': {'n_estimators': 909, 'max_depth': 3, 'learning_rate': 0.005},
    'turkish': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005}
}

best_params_ou25 = {
    'CH': {'n_estimators': 1105, 'max_depth': 3, 'learning_rate': 0.01},
    'belgian': {'n_estimators': 2481, 'max_depth': 3, 'learning_rate': 0.1},
    'british_champ': {'n_estimators': 2085, 'max_depth': 3, 'learning_rate': 0.005},
    'british_conference': {'n_estimators': 849, 'max_depth': 3, 'learning_rate': 0.005},
    'british_pl': {'n_estimators': 825, 'max_depth': 3, 'learning_rate': 0.005},
    'chn': {'n_estimators': 1331, 'max_depth': 3, 'learning_rate': 0.005},
    'dutch': {'n_estimators': 915, 'max_depth': 3, 'learning_rate': 0.005},
    'fin': {'n_estimators': 2543, 'max_depth': 3, 'learning_rate': 0.01},
    'french': {'n_estimators': 2017, 'max_depth': 3, 'learning_rate': 0.005},
    'german': {'n_estimators': 1195, 'max_depth': 3, 'learning_rate': 0.005},
    'german_2': {'n_estimators': 809, 'max_depth': 3, 'learning_rate': 0.005},
    'greek': {'n_estimators': 929, 'max_depth': 3, 'learning_rate': 0.005},
    'italian': {'n_estimators': 803, 'max_depth': 3, 'learning_rate': 0.005},
    'japan': {'n_estimators': 1189, 'max_depth': 3, 'learning_rate': 0.005},
    'norsk': {'n_estimators': 801, 'max_depth': 3, 'learning_rate': 0.005},
    'porto': {'n_estimators': 927, 'max_depth': 3, 'learning_rate': 0.005},
    'russian': {'n_estimators': 1963, 'max_depth': 3, 'learning_rate': 0.01},
    'spanish': {'n_estimators': 807, 'max_depth': 3, 'learning_rate': 0.005},
    'spanish_2': {'n_estimators': 813, 'max_depth': 3, 'learning_rate': 0.005},
    'swedish': {'n_estimators': 1045, 'max_depth': 3, 'learning_rate': 0.005},
    'turkish': {'n_estimators': 2211, 'max_depth': 3, 'learning_rate': 0.01}
}

best_params_btts = {
    'CH': {'n_estimators': 947, 'max_depth': 3, 'learning_rate': 0.005},
    'belgian': {'n_estimators': 803, 'max_depth': 3, 'learning_rate': 0.005},
    'british_champ': {'n_estimators': 955, 'max_depth': 3, 'learning_rate': 0.005},
    'british_conference': {'n_estimators': 1069, 'max_depth': 3, 'learning_rate': 0.005},
    'british_pl': {'n_estimators': 947, 'max_depth': 3, 'learning_rate': 0.005},
    'chn': {'n_estimators': 2705, 'max_depth': 3, 'learning_rate': 0.01},
    'dutch': {'n_estimators': 3209, 'max_depth': 3, 'learning_rate': 0.01},
    'fin': {'n_estimators': 1051, 'max_depth': 3, 'learning_rate': 0.005},
    'french': {'n_estimators': 3295, 'max_depth': 3, 'learning_rate': 0.01},
    'german': {'n_estimators': 997, 'max_depth': 3, 'learning_rate': 0.005},
    'german_2': {'n_estimators': 803, 'max_depth': 3, 'learning_rate': 0.005},
    'greek': {'n_estimators': 993, 'max_depth': 3, 'learning_rate': 0.01},
    'italian': {'n_estimators': 1637, 'max_depth': 3, 'learning_rate': 0.01},
    'japan': {'n_estimators': 1343, 'max_depth': 3, 'learning_rate': 0.01},
    'norsk': {'n_estimators': 1221, 'max_depth': 3, 'learning_rate': 0.005},
    'porto': {'n_estimators': 815, 'max_depth': 3, 'learning_rate': 0.005},
    'russian': {'n_estimators': 2089, 'max_depth': 3, 'learning_rate': 0.01},
    'spanish': {'n_estimators': 2399, 'max_depth': 3, 'learning_rate': 0.01},
    'spanish_2': {'n_estimators': 1123, 'max_depth': 3, 'learning_rate': 0.005},
    'swedish': {'n_estimators': 1407, 'max_depth': 3, 'learning_rate': 0.005},
    'turkish': {'n_estimators': 939, 'max_depth': 3, 'learning_rate': 0.005}
}

""" --- 1X2 Model Parameters ---
  - model_1x2_CH.joblib            | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_belgian.joblib       | n_estimators: 1109, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_champ.joblib | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_conference.joblib | n_estimators: 935, max_depth: 3, learning_rate: 0.005
  - model_1x2_british_pl.joblib    | n_estimators: 1315, max_depth: 3, learning_rate: 0.005
  - model_1x2_bundesliga_2.joblib  | n_estimators: 875, max_depth: 3, learning_rate: 0.01
  - model_1x2_chn.joblib           | n_estimators: 1143, max_depth: 3, learning_rate: 0.005
  - model_1x2_dutch.joblib         | n_estimators: 1185, max_depth: 3, learning_rate: 0.005
  - model_1x2_fin.joblib           | n_estimators: 1275, max_depth: 3, learning_rate: 0.005
  - model_1x2_french.joblib        | n_estimators: 849, max_depth: 3, learning_rate: 0.005
  - model_1x2_german.joblib        | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_german_2.joblib      | n_estimators: 831, max_depth: 3, learning_rate: 0.005
  - model_1x2_greek.joblib         | n_estimators: 803, max_depth: 3, learning_rate: 0.005
  - model_1x2_italian.joblib       | n_estimators: 809, max_depth: 3, learning_rate: 0.005
  - model_1x2_japan.joblib         | n_estimators: 849, max_depth: 3, learning_rate: 0.005
  - model_1x2_no_odds.joblib       | n_estimators: None, max_depth: None, learning_rate: None
  - model_1x2_norsk.joblib         | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_1x2_porto.joblib         | n_estimators: 1019, max_depth: 3, learning_rate: 0.01
  - model_1x2_russian.joblib       | n_estimators: 879, max_depth: 3, learning_rate: 0.005
  - model_1x2_spanish.joblib       | n_estimators: 879, max_depth: 3, learning_rate: 0.005
  - model_1x2_spanish_2.joblib     | n_estimators: 1811, max_depth: 3, learning_rate: 0.005
  - model_1x2_swedish.joblib       | n_estimators: 909, max_depth: 3, learning_rate: 0.005
  - model_1x2_tuned_no_odds.joblib | n_estimators: 33331, max_depth: 3, learning_rate: 0.001
  - model_1x2_turkish.joblib       | n_estimators: 801, max_depth: 3, learning_rate: 0.005

--- Over/Under 2.5 Model Parameters ---
  - model_ou25_CH.joblib           | n_estimators: 1105, max_depth: 3, learning_rate: 0.01
  - model_ou25_belgian.joblib      | n_estimators: 2481, max_depth: 3, learning_rate: 0.1
  - model_ou25_british_champ.joblib | n_estimators: 2085, max_depth: 3, learning_rate: 0.005
  - model_ou25_british_conference.joblib | n_estimators: 849, max_depth: 3, learning_rate: 0.005
  - model_ou25_british_pl.joblib   | n_estimators: 825, max_depth: 3, learning_rate: 0.005
  - model_ou25_bundesliga_2.joblib | n_estimators: 861, max_depth: 3, learning_rate: 0.005
  - model_ou25_chn.joblib          | n_estimators: 1331, max_depth: 3, learning_rate: 0.005
  - model_ou25_dutch.joblib        | n_estimators: 915, max_depth: 3, learning_rate: 0.005
  - model_ou25_fin.joblib          | n_estimators: 2543, max_depth: 3, learning_rate: 0.01
  - model_ou25_french.joblib       | n_estimators: 2017, max_depth: 3, learning_rate: 0.005
  - model_ou25_german.joblib       | n_estimators: 1195, max_depth: 3, learning_rate: 0.005
  - model_ou25_german_2.joblib     | n_estimators: 809, max_depth: 3, learning_rate: 0.005
  - model_ou25_greek.joblib        | n_estimators: 929, max_depth: 3, learning_rate: 0.005
  - model_ou25_italian.joblib      | n_estimators: 803, max_depth: 3, learning_rate: 0.005
  - model_ou25_japan.joblib        | n_estimators: 1189, max_depth: 3, learning_rate: 0.005
  - model_ou25_no_odds.joblib      | n_estimators: None, max_depth: None, learning_rate: None
  - model_ou25_norsk.joblib        | n_estimators: 801, max_depth: 3, learning_rate: 0.005
  - model_ou25_porto.joblib        | n_estimators: 927, max_depth: 3, learning_rate: 0.005
  - model_ou25_russian.joblib      | n_estimators: 1963, max_depth: 3, learning_rate: 0.01
  - model_ou25_spanish.joblib      | n_estimators: 807, max_depth: 3, learning_rate: 0.005
  - model_ou25_spanish_2.joblib    | n_estimators: 813, max_depth: 3, learning_rate: 0.005
  - model_ou25_swedish.joblib      | n_estimators: 1045, max_depth: 3, learning_rate: 0.005
  - model_ou25_turkish.joblib      | n_estimators: 2211, max_depth: 3, learning_rate: 0.01

--- BTTS Model Parameters ---
  - model_btts_CH.joblib           | n_estimators: 947, max_depth: 3, learning_rate: 0.005
  - model_btts_belgian.joblib      | n_estimators: 803, max_depth: 3, learning_rate: 0.005
  - model_btts_british_champ.joblib | n_estimators: 955, max_depth: 3, learning_rate: 0.005
  - model_btts_british_conference.joblib | n_estimators: 1069, max_depth: 3, learning_rate: 0.005
  - model_btts_british_pl.joblib   | n_estimators: 947, max_depth: 3, learning_rate: 0.005
  - model_btts_bundesliga_2.joblib | n_estimators: 905, max_depth: 3, learning_rate: 0.005
  - model_btts_chn.joblib          | n_estimators: 2705, max_depth: 3, learning_rate: 0.01
  - model_btts_dutch.joblib        | n_estimators: 3209, max_depth: 3, learning_rate: 0.01
  - model_btts_fin.joblib          | n_estimators: 1051, max_depth: 3, learning_rate: 0.005
  - model_btts_french.joblib       | n_estimators: 3295, max_depth: 3, learning_rate: 0.01
  - model_btts_german.joblib       | n_estimators: 997, max_depth: 3, learning_rate: 0.005
  - model_btts_german_2.joblib     | n_estimators: 803, max_depth: 3, learning_rate: 0.005
  - model_btts_greek.joblib        | n_estimators: 993, max_depth: 3, learning_rate: 0.01
  - model_btts_italian.joblib      | n_estimators: 1637, max_depth: 3, learning_rate: 0.01
  - model_btts_japan.joblib        | n_estimators: 1343, max_depth: 3, learning_rate: 0.01
  - model_btts_no_odds.joblib      | n_estimators: None, max_depth: None, learning_rate: None
  - model_btts_norsk.joblib        | n_estimators: 1221, max_depth: 3, learning_rate: 0.005
  - model_btts_porto.joblib        | n_estimators: 815, max_depth: 3, learning_rate: 0.005
  - model_btts_russian.joblib      | n_estimators: 2089, max_depth: 3, learning_rate: 0.01
  - model_btts_spanish.joblib      | n_estimators: 2399, max_depth: 3, learning_rate: 0.01
  - model_btts_spanish_2.joblib    | n_estimators: 1123, max_depth: 3, learning_rate: 0.005
  - model_btts_swedish.joblib      | n_estimators: 1407, max_depth: 3, learning_rate: 0.005
  - model_btts_turkish.joblib      | n_estimators: 939, max_depth: 3, learning_rate: 0.005


--- All specialized xG models have been trained and saved. ---"""

# --- Generate "Out-of-Fold" predictions to create features for the Meta-Model ---
print("\nGenerating meta-features... This may take some time.")
meta_features = []
leagues = all_data['league'].unique()
feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']

for league in leagues:
    print(f"  Processing meta-features for {league.title()}...")
    league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols).copy()
    X_league = league_data[feature_cols]
    
    # Process 1X2
    if league in best_params_1x2:
        model_template = CalibratedClassifierCV(estimator=XGBClassifier(objective='multi:softprob', **best_params_1x2[league]), cv=3)
        probs_1x2 = cross_val_predict(model_template, X_league, league_data['result_label'], cv=5, method='predict_proba', n_jobs=-1)
        league_data['pred_H'], league_data['pred_D'], league_data['pred_A'] = probs_1x2[:, 2], probs_1x2[:, 1], probs_1x2[:, 0]

    # Process O/U 2.5
    if league in best_params_ou25:
        model_template = CalibratedClassifierCV(estimator=XGBClassifier(objective='binary:logistic', **best_params_ou25[league]), cv=3)
        probs_ou = cross_val_predict(model_template, X_league, league_data['over_2_5'], cv=5, method='predict_proba', n_jobs=-1)
        league_data['pred_O25'], league_data['pred_U25'] = probs_ou[:, 1], probs_ou[:, 0]
        
    # Process BTTS
    if league in best_params_btts:
        model_template = CalibratedClassifierCV(estimator=XGBClassifier(objective='binary:logistic', **best_params_btts[league]), cv=3)
        probs_btts = cross_val_predict(model_template, X_league, league_data['btts'], cv=5, method='predict_proba', n_jobs=-1)
        league_data['pred_BTTS_Y'], league_data['pred_BTTS_N'] = probs_btts[:, 1], probs_btts[:, 0]
    
    meta_features.append(league_data)

meta_data = pd.concat(meta_features, ignore_index=True)
print("Meta-features generated successfully.")

# --- Train the final Meta-Model ---
print("\nTraining the final Meta-Model...")
meta_feature_cols = ['pred_H', 'pred_D', 'pred_A', 'pred_O25', 'pred_U25', 'pred_BTTS_Y', 'pred_BTTS_N']
meta_data_final = meta_data.dropna(subset=meta_feature_cols)
meta_X = meta_data_final[meta_feature_cols]
meta_y = meta_data_final['result_label']
meta_model = LogisticRegression(multi_class='ovr').fit(meta_X, meta_y)
print("Meta-Model training complete.")

# --- Save the Entire Pipeline ---
pipeline = {
    'base_models_1x2': {lg: joblib.load(f'model_1x2_{lg}.joblib') for lg in leagues if lg in best_params_1x2},
    'base_models_ou25': {lg: joblib.load(f'model_ou25_{lg}.joblib') for lg in leagues if lg in best_params_ou25},
    'base_models_btts': {lg: joblib.load(f'model_btts_{lg}.joblib') for lg in leagues if lg in best_params_btts},
    'meta_model': meta_model,
    'historical_data': pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
}
joblib.dump(pipeline, 'full_prediction_pipeline.joblib')
print("\n--- Complete prediction pipeline saved to 'full_prediction_pipeline.joblib' ---")

Successfully loaded processed data.

Generating meta-features... This may take some time.
  Processing meta-features for Japan...
  Processing meta-features for Norsk...
  Processing meta-features for Swedish...
  Processing meta-features for Fin...
  Processing meta-features for Ch...
  Processing meta-features for Russian...
  Processing meta-features for Chn...
  Processing meta-features for Belgian...
  Processing meta-features for French...
  Processing meta-features for Turkish...
  Processing meta-features for Greek...
  Processing meta-features for British_Champ...
  Processing meta-features for Dutch...
  Processing meta-features for British_Pl...
  Processing meta-features for Spanish_2...
  Processing meta-features for Spanish...
  Processing meta-features for German...
  Processing meta-features for German_2...
  Processing meta-features for Porto...
  Processing meta-features for Italian...
  Processing meta-features for British_Conference...
  Processing meta-features for

In [36]:
# ==============================================================================
# SCRIPT TO VALIDATE THE FINAL META-MODEL
# ==============================================================================
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, log_loss
import warnings
warnings.filterwarnings('ignore')

try:
    # Load the entire saved pipeline and data
    pipeline = joblib.load('full_prediction_pipeline.joblib')
    all_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])

    # Unpack the components
    models_1x2 = pipeline['base_models_1x2']
    models_ou25 = pipeline['base_models_ou25']
    models_btts = pipeline['base_models_btts']
    meta_model = pipeline['meta_model']

    print("--- Preparing test data for meta-model validation ---")

    # Define feature columns and create the test set (newest 15% of data)
    feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
    split_index = int(len(all_data) * 0.85)
    test_data = all_data.iloc[split_index:].copy()

    # --- Generate Meta-Features for the Test Set ---
    meta_features_test = []
    leagues = test_data['league'].unique()

    for league in leagues:
        league_test_data = test_data[test_data['league'] == league].dropna(subset=feature_cols).copy()
        X_league_test = league_test_data[feature_cols]
        
        if league in models_1x2:
            probs_1x2 = models_1x2[league].predict_proba(X_league_test)
            league_test_data['pred_H'], league_test_data['pred_D'], league_test_data['pred_A'] = probs_1x2[:, 2], probs_1x2[:, 1], probs_1x2[:, 0]
        
        if league in models_ou25:
            probs_ou = models_ou25[league].predict_proba(X_league_test)
            league_test_data['pred_O25'], league_test_data['pred_U25'] = probs_ou[:, 1], probs_ou[:, 0]
            
        if league in models_btts:
            probs_btts = models_btts[league].predict_proba(X_league_test)
            league_test_data['pred_BTTS_Y'], league_test_data['pred_BTTS_N'] = probs_btts[:, 1], probs_btts[:, 0]
        
        meta_features_test.append(league_test_data)

    meta_test_data = pd.concat(meta_features_test, ignore_index=True)
    
    # --- Make Final Predictions and Evaluate ---
    meta_feature_cols = ['pred_H', 'pred_D', 'pred_A', 'pred_O25', 'pred_U25', 'pred_BTTS_Y', 'pred_BTTS_N']
    meta_test_final = meta_test_data.dropna(subset=meta_feature_cols)

    meta_X_test = meta_test_final[meta_feature_cols]
    meta_y_test = meta_test_final['result_label']

    final_preds = meta_model.predict(meta_X_test)
    final_probs = meta_model.predict_proba(meta_X_test)

    accuracy = accuracy_score(meta_y_test, final_preds)
    loss = log_loss(meta_y_test, final_probs)

    print("\n--- Final Meta-Model Performance (on unseen test data) ---")
    print(f"Overall Accuracy: {accuracy:.2%}")
    print(f"Overall Log Loss: {loss:.4f}")

except FileNotFoundError:
    print("Could not find the 'full_prediction_pipeline.joblib' or data files.")

    # ==============================================================================
# ADVANCED SCRIPT TO VALIDATE 1X2 MODEL PERFORMANCE
# ==============================================================================
import pandas as pd
import glob
import joblib
import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    # Load the master dataset that was used for training
    all_data = pd.read_csv('full_processed_data.csv', parse_dates=['Date'])
    
    # Get a list of all leagues for which a 1X2 model was saved
    model_files = sorted(glob.glob('model_1x2_*.joblib'))
    leagues = [f.replace('model_1x2_', '').replace('.joblib', '') for f in model_files]
    
    print(f"Found {len(leagues)} specialized 1X2 models to test.\n")
    
    # --- Define the feature columns used for training ---
    feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
    
    print(f"{'League':<20} | {'Top-1 Acc':<12} | {'Top-2 Acc':<12} | {'Upset Freq':<12}")
    print("-" * 65)

    # --- Loop through each league, load its model, and test it ---
    for league in leagues:
        model = joblib.load(f'model_1x2_{league}.joblib')
        league_data = all_data[all_data['league'] == league].dropna(subset=feature_cols + ['result_label'])
        
        # Perform the same 85/15 train/test split to get the correct test set
        X, y = league_data[feature_cols], league_data['result_label']
        split_index = int(len(X) * 0.85)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        if len(X_test) > 0:
            # Get predicted probabilities
            y_pred_proba = model.predict_proba(X_test)
            
            # Get the top 1, 2, and 3 predictions for each match
            # argsort sorts from smallest to largest, so we reverse it with [::-1]
            sorted_preds = np.argsort(y_pred_proba, axis=1)[:, ::-1]
            
            top_1_prediction = sorted_preds[:, 0]
            top_2_predictions = sorted_preds[:, :2]
            top_3_prediction = sorted_preds[:, 2]

            # Calculate metrics
            top_1_correct = np.sum(top_1_prediction == y_test.values)
            top_2_correct = np.sum([y_test.values[i] in top_2_predictions[i] for i in range(len(y_test))])
            top_3_correct = np.sum(top_3_prediction == y_test.values)

            total_matches = len(y_test)
            top_1_acc = top_1_correct / total_matches
            top_2_acc = top_2_correct / total_matches
            upset_freq = top_3_correct / total_matches
            
            print(f"{league.title():<20} | {top_1_acc:<12.2%} | {top_2_acc:<12.2%} | {upset_freq:<12.2%}")

except FileNotFoundError:
    print("Could not find the 'full_processed_data.csv' or model files. Make sure they are in the correct directory.")

--- Preparing test data for meta-model validation ---

--- Final Meta-Model Performance (on unseen test data) ---
Overall Accuracy: 59.98%
Overall Log Loss: 0.8760
Found 24 specialized 1X2 models to test.

League               | Top-1 Acc    | Top-2 Acc    | Upset Freq  
-----------------------------------------------------------------
Ch                   | 53.94%       | 82.42%       | 17.58%      
Belgian              | 61.80%       | 88.41%       | 11.59%      
British_Champ        | 54.35%       | 84.06%       | 15.94%      
British_Conference   | 53.03%       | 79.80%       | 20.20%      
British_Pl           | 61.05%       | 86.67%       | 13.33%      
Bundesliga_2         | 61.96%       | 91.85%       | 8.15%       
Chn                  | 61.38%       | 88.89%       | 11.11%      
Dutch                | 63.48%       | 90.87%       | 9.13%       
Fin                  | 63.43%       | 92.54%       | 7.46%       
French               | 63.12%       | 86.31%       | 13.69%      
Ge

In [43]:
# ==============================================================================
# FINAL INTERACTIVE PREDICTOR (USING THE META-MODEL)
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from IPython.display import display, HTML
from ipywidgets import widgets, interactive_output, VBox

# --- Load the entire saved pipeline ---
try:
    pipeline = joblib.load('full_prediction_pipeline.joblib')
    # Unpack the components from the pipeline
    models_1x2 = pipeline['base_models_1x2']
    models_ou25 = pipeline['base_models_ou25']
    models_btts = pipeline['base_models_btts']
    meta_model = pipeline['meta_model']
    historical_data = pipeline['historical_data']
    # Prepare team list and league map for the UI
    all_teams = sorted(pd.concat([historical_data['HomeTeam'], historical_data['AwayTeam']]).unique())
    team_to_league_map = historical_data.drop_duplicates(subset=['HomeTeam', 'league']).set_index('HomeTeam')['league'].to_dict()
    away_map = historical_data.drop_duplicates(subset=['AwayTeam', 'league']).set_index('AwayTeam')['league'].to_dict()
    team_to_league_map.update(away_map)
    print("--- Prediction Environment Ready ---")
    print("Models loaded. Please select teams below.")
except FileNotFoundError:
    print("--- ERROR: 'full_prediction_pipeline.joblib' not found. ---")
    print("--- Please run the main training script for the meta-model first. ---")

# --- Create the Master Stats Table (for live stat lookups) ---
final_stats_list = []
for team_name in all_teams:
    overall_matches = historical_data[(historical_data['HomeTeam'] == team_name) | (historical_data['AwayTeam'] == team_name)]
    if overall_matches.empty: continue
    last_match_overall = overall_matches.iloc[-1]
    if last_match_overall['HomeTeam'] == team_name:
        elo, form = last_match_overall['FinalHomeElo'], last_match_overall['HomeForm']
    else:
        elo, form = last_match_overall['FinalAwayElo'], last_match_overall['AwayForm']
    final_stats_list.append({'Team': team_name, 'FinalElo': elo, 'FinalForm': form})
final_team_stats = pd.DataFrame(final_stats_list).set_index('Team')

# --- The Prediction Function ---
def predict_live_match(home_team, away_team):
    if not home_team or not away_team or home_team == away_team: return 
    
    try:
        league = team_to_league_map.get(home_team)
        if not league:
            display(HTML(f"<p style='color:red;'>Could not determine league for {home_team}.</p>"))
            return
        # Get the base models for the correct league
        model_1x2 = models_1x2[league]
        model_ou25 = models_ou25[league]
        model_btts = models_btts[league]
        
        home_stats = final_team_stats.loc[home_team]
        away_stats = final_team_stats.loc[away_team]
        
    except (IndexError, FileNotFoundError, KeyError):
        display(HTML(f"<p style='color:red;'>Could not load models or find stats. Ensure models for '{league}' were trained.</p>"))
        return
        
    # Create the feature vector for the base models
    feature_cols = ['Elo_diff', 'HomeForm', 'AwayForm', 'AvgHomeGoals', 'AvgAwayGoals', 'AvgHomeConceded', 'AvgAwayConceded']
    # NOTE: Goal averages are not used by the model but are needed for the dataframe structure
    fixture_df = pd.DataFrame([{'Elo_diff': home_stats['FinalElo'] - away_stats['FinalElo'], 'HomeForm': home_stats['FinalForm'], 'AwayForm': away_stats['FinalForm'],
                                'AvgHomeGoals': 0, 'AvgAwayGoals': 0, 'AvgHomeConceded': 0, 'AvgAwayConceded': 0}])[feature_cols]

    # --- Get predictions from the base "expert" models ---
    probs_1x2 = model_1x2.predict_proba(fixture_df)[0]
    probs_ou = model_ou25.predict_proba(fixture_df)[0]
    probs_btts = model_btts.predict_proba(fixture_df)[0]
    
    # --- Create the feature vector for the "Chief Doctor" meta-model ---
    meta_features = pd.DataFrame([{'pred_H': probs_1x2[2], 'pred_D': probs_1x2[1], 'pred_A': probs_1x2[0],
                                   'pred_O25': probs_ou[1], 'pred_U25': probs_ou[0],
                                   'pred_BTTS_Y': probs_btts[1], 'pred_BTTS_N': probs_btts[0]}])
    
    # --- Get the final, wisest prediction from the meta-model ---
    final_probs = meta_model.predict_proba(meta_features)[0]
        
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 10px; margin-top: 10px;">
    <h3>Final Meta-Model Prediction: {home_team} vs {away_team}</h3>
    <p><i>(Using the "Chief Doctor" model to judge the specialist opinions for the <b>{league.title()}</b> league)</i></p>
    <table border="1" style="width:100%; text-align:center;">
      <tr style="background-color:#f2f2f2;"><th>Home Win</th><th>Draw</th><th>Away Win</th></tr>
      <tr style="font-size: 1.2em; font-weight: bold;">
        <td>{final_probs[2]:.1%}</td>
        <td>{final_probs[1]:.1%}</td>
        <td>{final_probs[0]:.1%}</td>
      </tr>
    </table></div>"""))

# --- Create and Display the Interactive Dropdown Widgets ---
style = {'description_width': 'initial'}
home_dropdown = widgets.Dropdown(options=all_teams, description='Home Team:', style=style, layout={'width': '500px'})
away_dropdown = widgets.Dropdown(options=all_teams, description='Away Team:', style=style, layout={'width': '500px'}, value=all_teams[1] if len(all_teams) > 1 else None)

ui = VBox([home_dropdown, away_dropdown])
out = widgets.interactive_output(predict_live_match, {'home_team': home_dropdown, 'away_team': away_dropdown})

display(ui, out)


--- Prediction Environment Ready ---
Models loaded. Please select teams below.


Output()